# Suite LIF — Lifetimes: every read within a transaction

Every read of a store's data is made within a transaction, as every write is ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md#decided)). A visitor of the store's
data, an object and each visitor read from it (its properties, its adjacencies, its entries, the value objects and
containers it holds), lives only within the transaction that made it, the outermost open on its store; used after it
ends, it is refused. The store lives on, and so do a path, a reference and every value a program copies out within a
transaction, which is stale from the moment the transaction ends. Within a transaction, the store gives one visitor of
each object. The durable object the store holds is wrapped by each transaction's visitor of it, which reads and
writes it through the transaction's proposals. A frame `store.transaction` gives back holds bindings and assignments,
not visitors: each symbol's object by its reference, and the values the transaction left in the properties its specs
name; it refuses a counted object, which has no reference. Schema objects are not covered.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Plain, Proxies, References
from support import properties_of, raises


def crm(txn):
    """crm.Person, owned by the singleton crm.Book by key, with a home address (a value object) and tags (a container);
    crm.Note, counted, about a person; crm.Tag, counted, linking nothing."""
    S = txn.Schemas
    return txn.uid("lif/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Address": S.OfObject, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject, "About": S.OfRelation, "Tag": S.OfObject})
        .spec([
            lambda _: _.Address.name("crm.Address").properties({"city": S.OfStr}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({
                "name": S.OfStr, "home": _.Address,
                "tags": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr]))}).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True)]),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
            lambda _: _.Tag.name("crm.Tag").ref(True).properties({"label": S.OfStr}),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book, Note, Tag = (store.registered(name) for name in ("crm.Person", "crm.Book", "crm.Note", "crm.Tag"))
store.transaction(lambda t: t.uid("people").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"book": Book, "ada": Person, "bob": Person}).spec([
        lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada), "bob": lambda e: e.contact(_.bob)}),
        lambda _: _.ada.name("Ada"),
        lambda _: _.bob.name("Bob"),
    ])]))]))
ADA = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "ada"},
                                                       "next": {"target": {}}}}}


def refused(use):
    """Asserts that `use`, a use of a visitor whose transaction has ended, is refused."""
    with raises(ValueError, match="a visitor lives only within the transaction that made it"):
        use()

<a id="LIF-01"></a>
## LIF-01 · A visitor lives within the transaction that made it

An object, its property's visitor, its adjacency and an entry of it, read within a transaction, are refused once it
ends, each whatever the use: a member read, a conversion, iteration, its length, its representation. What the program
copied out within it is its own.

In [ ]:
with store.transaction():
    book = store.singleton("crm.TheBook")
    contacts = book.contacts
    entry = contacts.at("ada")
    ada = entry.contact
    name = ada.name
    harvested = str(name)
    assert harvested == "Ada" and contacts.at("ada").contact is ada and contacts[0] is entry  # one visitor of each
for use in (lambda: book.contacts, lambda: ada.name, lambda: str(name), lambda: name.get(), lambda: entry.contact,
            lambda: entry.key, lambda: list(contacts), lambda: len(contacts), lambda: contacts[0], lambda: repr(ada),
            lambda: ada.as_schema()):
    refused(use)
assert harvested == "Ada"  # stale from the moment the transaction ended, but the program's

<a id="LIF-02"></a>
## LIF-02 · A store is read only within a transaction

`store.singleton(...)` and `store.extent(...)` are refused with no transaction open. Each transaction gives visitors of
its own: the singleton again, another visitor of the same object, the earlier one refused. A visitor of a transaction
given to a later one, to update it or to link it, is refused.

In [ ]:
for use in (lambda: store.singleton("crm.TheBook"), lambda: store.extent("crm.Person")):
    with raises(ValueError, match="a store is read only within a transaction (store.transaction)"):
        use()
with store.transaction():
    again = store.singleton("crm.TheBook")
    assert again is not book and again is store.singleton("crm.TheBook")  # one visitor of it, in this transaction
    assert sorted(str(person.name) for person in store.extent("crm.Person")) == ["Ada", "Bob"]
    with raises(ValueError, match="a visitor lives only within the transaction that made it"):
        store.crm.Person(ada, lambda p: p.name("Ada Lovelace"))  # a visitor of the earlier transaction
    with raises(ValueError, match="a visitor lives only within the transaction that made it"):
        store.crm.Note(lambda n: n.about([ada]))

<a id="LIF-03"></a>
## LIF-03 · A value object and its visitor

A value object held by nothing, built standalone, is data: read anywhere, and given to an object, copied, so that it
stays the program's. Read through the object that holds it, a value object, or a container, is a visitor of the
transaction, owned by that object, and refused once the transaction ends.

In [ ]:
paris = store.builder("crm.Address").city("Paris").create()  # no transaction: a standalone value, data
assert str(paris.city) == "Paris" and Proxies.owner_of(paris) is None
with store.transaction():
    ada = store.singleton("crm.TheBook").contacts.at("ada").contact
    store.crm.Person(ada, lambda p: p.home(paris).tags(["math", "poetry"]))
    homes, tags = [], []
    ada.home.as_object(lambda home: homes.append(home) or home)
    ada.tags.as_value(lambda a: a.as_container(lambda held: tags.append(held) or held))
    (home,), (held,) = homes, tags
    assert str(home.city) == "Paris" and home is not paris and Proxies.owner_of(home) is ada  # the object's copy
    assert len(held) == 2 and Proxies.owner_of(paris) is None  # the value given stays the program's
for use in (lambda: home.city, lambda: len(held), lambda: Proxies.owner_of(home)):
    refused(use)
assert str(paris.city) == "Paris"  # data, read with no transaction open

<a id="LIF-04"></a>
## LIF-04 · Frames hold bindings and the state they reference, not visitors

A frame `store.transaction` gives back is its journal ([CAP-318: support for decoupled code](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-318)), a value read anywhere: its
symbols, each its object's reference, which the frame gives by symbol, and the state its path's clauses reference, the
values the transaction left in the properties its specs name among it. A symbol bound to a counted object, which has
no reference, is refused, and left out of the journal. `References.of` gives a frame's reference, its journal among
it, the counted object annotated on it as an error.


In [ ]:
(frame,) = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"who": Person}).spec([lambda _: _.who.name("Ada")])]))]))
reference = frame["who"]
assert Plain.ToPlain(store)(reference.as_schema(), reference) == {"path": ADA}  # read with no transaction open
assert properties_of(store, frame) == {"who": {"name": "Ada"}} and frame.errors == []
(noted,) = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p
    .symbols({"who": Person}).spec([lambda _: _.who.name("Ada")])]).then([lambda n: n.upsert(lambda u: u.key("n/1")
    .exists([lambda p: p.symbols({"who": Person, "note": Note}).spec([lambda _: _.note.text("hi").about([_.who])])]))]))]))
assert list(noted) == ["who", "note"] and properties_of(store, noted) == {"who": {"name": "Ada"}}
held = list(noted.values())[1]  # what the frame holds of it: refused whatever the use
assert repr(held) == "<refused: a counted object>"
with raises(LookupError, match="a counted 'crm.Note' has no reference"):
    held.text
for use in (lambda: noted["note"], lambda: noted.get("note")):
    with raises(LookupError, match="symbol 'note' of the frame: a counted 'crm.Note' has no reference"):
        use()
written = References.of(store, frame)
written = Plain.ToPlain(store)(written.as_schema(), written)
assert written["scope"] == "0" and written["journal"]["symbols"] == {"who": {"reference": {"path": ADA}}}  # its input frame's id
assert written["journal"]["changes"] == [{"property": {"set": {"holder": "who", "name": "name", "value": "Ada"}}}]
written = References.of(store, noted)
assert [error["message"][:61] for error in Plain.ToPlain(store)(written.as_schema(), written)["errors"]] == [
    "symbol 'note' of the frame: a counted 'crm.Note' has no refer"]

<a id="LIF-05"></a>
## LIF-05 · What a decoder gives back

A document read within a transaction gives visitors of it; read with none open, the decoder's own transaction makes
them, and what it gives back is refused.

In [ ]:
with store.transaction():
    document = Plain.ToPlain(store).Reachable(Tag, store.crm.Tag(lambda t: t.label("urgent")))
read = Plain.FromPlain(store).Reachable(Tag, document)  # its own transaction, ended
refused(lambda: read.label)
with store.transaction():
    assert str(Plain.FromPlain(store).Reachable(Tag, document).label) == "urgent"

<a id="LIF-06"></a>
## LIF-06 · Expiry follows the transaction that made a visitor

A transaction's expression built with no transaction open is no transaction's, and lives on; one built within a
transaction is that transaction's, and is refused once it ends, whatever its class.

In [ ]:
expression = store.Mutations.OfTransaction(lambda t: t.uid("later"))  # built with none open
with store.transaction():
    within = store.Mutations.OfTransaction(lambda t: t.uid("within"))
    assert str(within.uid) == "within"
assert str(expression.uid) == "later"
refused(lambda: within.uid)
store.transaction(expression)  # executed later